# 📓 Notebook 1 — Chuẩn bị dữ liệu

**Đề tài**: Phân loại rau sạch / rau hỏng bằng Deep Learning

Notebook này thực hiện:
1. Kiểm tra môi trường (Python, TF, GPU)
2. Cài đặt dependencies
3. Tải dataset **Freshness44** từ Kaggle (15.000 ảnh)
4. Sắp xếp về `dataset/raw/{fresh,rotten}/`
5. Kiểm tra chất lượng dataset (file lỗi, kích thước, đa dạng loại quả)
6. Tiền xử lý: resize 224×224, split train/valid/test 70/15/15
7. Trực quan hoá: ảnh mẫu, phân bố class, augmentation demo

Sau khi chạy xong notebook này → tiếp tục với **`02_train_and_evaluate.ipynb`**.

## 1. Thiết lập đường dẫn project

In [ ]:
import os, sys
from pathlib import Path

# Đảm bảo CWD là thư mục project (1 cấp trên notebook/)
ROOT = Path('..').resolve()
if Path.cwd().name == 'notebook':
    os.chdir(ROOT)
sys.path.insert(0, str(Path.cwd()))

print('📁 CWD :', Path.cwd())
print('🐍 Py  :', sys.version.split()[0])

## 2. Cài đặt dependencies

Chỉ chạy lần đầu — nếu đã `pip install -r requirements.txt` rồi thì bỏ qua.

In [ ]:
# Bỏ comment để cài (lần đầu)
# !pip install -q -r requirements.txt

import tensorflow as tf
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image

print('TF      :', tf.__version__)
print('NumPy   :', np.__version__)
print('GPU     :', tf.config.list_physical_devices('GPU') or '(CPU only)')

## 3. Tải dataset Freshness44 (Kaggle)

- **Nguồn**: https://www.kaggle.com/datasets/siavash93/freshness44
- **Kích thước gốc**: 53.616 ảnh, ~6.7 GB
- **Lấy mẫu**: 7500 ảnh / class = **15.000 ảnh**
- **Cache**: lưu vào ổ D (`D:/kaggle_cache`) để không đầy ổ C

💡 Lần đầu sẽ mở browser yêu cầu đăng nhập Kaggle.

In [ ]:
# Bỏ comment dòng dưới để chạy. Tốn ~15-30 phút lần đầu.
# !python tools/prepare_freshness44.py --cache-dir "D:/kaggle_cache" --max-per-class 7500

## 4. Kiểm tra dataset đã tải

In [ ]:
from collections import Counter
RAW = Path('dataset/raw')
assert RAW.exists(), 'Chưa có dataset/raw — chạy cell tải Freshness44 trước'

stats = {d.name: sum(1 for f in d.iterdir() if f.is_file())
         for d in RAW.iterdir() if d.is_dir()}
print('📦 Số ảnh từng class:')
for k, v in stats.items():
    print(f'   {k:<8s}: {v:,}')
print(f'   {"TỔNG":<8s}: {sum(stats.values()):,}')

In [ ]:
# Sanity check: scan toàn bộ ảnh, tìm file hỏng / nhỏ / sai định dạng
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

broken, small, sizes, types = [], [], [], Counter()
for cls_dir in sorted(d for d in RAW.iterdir() if d.is_dir()):
    for f in tqdm(list(cls_dir.iterdir()), desc=cls_dir.name):
        try:
            with Image.open(f) as im:
                w, h = im.size
                sizes.append((w, h))
                if min(w, h) < 64: small.append(f)
        except (UnidentifiedImageError, OSError):
            broken.append(f)
        # đoán loại quả từ tên file: <type>_<idx>.jpg
        types[f.stem.split('_')[0]] += 1

print(f'\n✅ Tổng ảnh quét  : {len(sizes):,}')
print(f'❌ Ảnh hỏng       : {len(broken)}')
print(f'⚠️  Ảnh < 64px    : {len(small)}')
print(f'🍎 Số loại quả    : {len(types)}')
print(f'\n🍎 Top 10 loại nhiều nhất:')
for t, n in types.most_common(10):
    print(f'   {t:<15s}: {n:,}')

In [ ]:
# Biểu đồ phân bố class + kích thước ảnh
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 4a. Phân bố class
bars = axes[0].bar(stats.keys(), stats.values(), color=['#2E7D32', '#C62828'])
for b, v in zip(bars, stats.values()):
    axes[0].text(b.get_x()+b.get_width()/2, v, f'{v:,}', ha='center', va='bottom')
axes[0].set_title('Phân bố số ảnh theo class', fontweight='bold')
axes[0].set_ylabel('Số ảnh')

# 4b. Histogram kích thước (chỉ lấy chiều rộng để đơn giản)
ws = [w for w, h in sizes]
axes[1].hist(ws, bins=40, color='#1976D2', edgecolor='white')
axes[1].set_title('Phân bố chiều rộng ảnh (px)', fontweight='bold')
axes[1].set_xlabel('Chiều rộng (px)')
axes[1].set_ylabel('Số ảnh')
axes[1].axvline(224, color='red', ls='--', label='Target 224')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Hiển thị 8 ảnh mẫu cho mỗi class
import random
random.seed(42)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for row, cls in enumerate(['fresh', 'rotten']):
    files = random.sample(list((RAW / cls).iterdir()), 8)
    for col, f in enumerate(files):
        axes[row, col].imshow(Image.open(f))
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls.upper(), loc='left',
                                     fontweight='bold', fontsize=12,
                                     color='#2E7D32' if cls=='fresh' else '#C62828')
plt.suptitle('Ảnh mẫu mỗi class', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Tiền xử lý + split train/valid/test

- Resize tất cả ảnh về **224 × 224**
- Convert RGB, lưu JPEG quality 92
- Split stratified theo class: **train 70% / valid 15% / test 15%**

💡 Tốn ~3-5 phút với 15k ảnh.

In [ ]:
!python preprocessing/preprocess.py --src dataset/raw --dst dataset --img-size 224

In [ ]:
# Verify các thư mục split
split_stats = {}
for split in ('train', 'valid', 'test'):
    sd = Path(f'dataset/{split}')
    if not sd.exists(): continue
    split_stats[split] = {c.name: sum(1 for _ in c.iterdir())
                          for c in sd.iterdir() if c.is_dir()}

df_split = pd.DataFrame(split_stats).fillna(0).astype(int)
print('📊 Phân bố sau split:')
print(df_split)
print(f'\nTỔNG: {df_split.values.sum():,} ảnh')

In [ ]:
# Bar chart: phân bố train/valid/test cho 2 class
ax = df_split.plot.bar(figsize=(8, 4),
                       color={'train': '#1976D2', 'valid': '#FFA726', 'test': '#7B1FA2'},
                       edgecolor='white')
ax.set_title('Phân bố train / valid / test', fontweight='bold')
ax.set_ylabel('Số ảnh'); ax.set_xlabel('Class')
ax.set_xticklabels(df_split.index, rotation=0)
for c in ax.containers:
    ax.bar_label(c, label_type='edge', fontsize=8)
plt.tight_layout(); plt.show()

## 6. Demo Augmentation

Hiển thị 8 phiên bản augmentation từ 1 ảnh gốc — kiểm tra xem augmentation có quá nặng không.

In [ ]:
from preprocessing.augmentation import build_train_generator

gen = build_train_generator(Path('dataset/train'), img_size=224, batch_size=1)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for ax in axes.ravel():
    x, _ = next(gen)
    ax.imshow(x[0])
    ax.axis('off')
plt.suptitle('8 phiên bản sau augmentation (rotation, flip, zoom, brightness, shift)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## ✅ Hoàn thành chuẩn bị

Đến đây bạn đã có:
- `dataset/raw/{fresh,rotten}/` — 15.000 ảnh gốc
- `dataset/processed/` — đã resize 224×224
- `dataset/train/` `valid/` `test/` — đã split 70/15/15
- `results/sample_images.png`, `results/split_distribution.png` — biểu đồ

👉 **Tiếp theo**: mở `02_train_and_evaluate.ipynb` để train mô hình và xem kết quả.